In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
import warnings
from sklearn.metrics import precision_score, recall_score, f1_score
import gc # Dọn dẹp bộ nhớ

warnings.filterwarnings('ignore')

# --- CẤU HÌNH ---
DATASET_ROOT = '/kaggle/input/vietnamese-foods' 
IMG_ROOT_DIR = os.path.join(DATASET_ROOT, 'Images')
if not os.path.exists(IMG_ROOT_DIR):
    DATASET_ROOT = '/kaggle/input/vietnam-food-dataset' # Fallback
    IMG_ROOT_DIR = os.path.join(DATASET_ROOT, 'Images')

TRAIN_DIR = os.path.join(IMG_ROOT_DIR, 'Train')
VAL_DIR = os.path.join(IMG_ROOT_DIR, 'Validate')
TEST_DIR = os.path.join(IMG_ROOT_DIR, 'Test')
IMG_SIZE = (128, 128) # Resize nhỏ để HOG tính nhanh

print(f"Đã cấu hình đường dẫn: {IMG_ROOT_DIR}")

In [ ]:
# Hàm lấy danh sách file ảnh
def get_image_paths(data_dir):
    paths = []
    labels = []
    if not os.path.exists(data_dir): return [], []
    
    for label in sorted(os.listdir(data_dir)):
        path = os.path.join(data_dir, label)
        if os.path.isdir(path):
            for img_name in os.listdir(path):
                paths.append(os.path.join(path, img_name))
                labels.append(label)
    return paths, labels

# Hàm tính HOG cho 1 ảnh
def compute_hog_single(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None: return None
        
        # 1. Resize & Grayscale
        img = cv2.resize(img, IMG_SIZE)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 2. Tính HOG
        # Ta chốt cứng tham số này để tránh tính đi tính lại nhiều lần
        fd = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                 cells_per_block=(2, 2), transform_sqrt=True,
                 block_norm='L2-Hys', feature_vector=True,
                channel_axis=-1)
        return fd
    except:
        return None

# Hàm xử lý cả bộ dữ liệu
def process_dataset_hog(dirs):
    all_paths = []
    all_labels = []
    for d in dirs:
        p, l = get_image_paths(d)
        all_paths.extend(p)
        all_labels.extend(l)
        
    print(f" Đang tính HOG cho {len(all_paths)} ảnh... (Vui lòng đợi)")
    
    # Chạy song song 4 luồng (n_jobs=4)
    features = Parallel(n_jobs=4)(delayed(compute_hog_single)(p) for p in tqdm(all_paths))
    
    # Lọc bỏ ảnh lỗi
    X, y = [], []
    for f, l in zip(features, all_labels):
        if f is not None:
            X.append(f)
            y.append(l)
            
    return np.array(X), np.array(y)

# --- THỰC HIỆN ---
# Gộp Train và Validate để train
print("--- XỬ LÝ TẬP TRAIN ---")
X_train_hog, y_train_raw = process_dataset_hog([TRAIN_DIR, VAL_DIR])

print("\n--- XỬ LÝ TẬP TEST ---")
X_test_hog, y_test_raw = process_dataset_hog([TEST_DIR])

print(f"\n Kích thước X_train: {X_train_hog.shape}")
print(f"Kích thước X_test:  {X_test_hog.shape}")

In [ ]:
# 1. Encode nhãn
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_raw)
y_test_enc = le.transform(y_test_raw)

# 2. Cấu hình tham số Random Forest
# Đây là các tham số bạn yêu cầu (đã bỏ bớt vài giá trị để chạy nhanh hơn cho 25k ảnh)
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1  # Sử dụng tất cả các luồng CPU để train nhanh hơn
)

rf_model.fit(X_train_hog, y_train_enc)
print("Huấn luyện hoàn tất.")


# --- 3. DỰ ĐOÁN VÀ ĐÁNH GIÁ ---
print("Đang dự đoán trên tập Test...")
y_pred = rf_model.predict(X_test_hog)

# Chuyển ngược lại từ số sang tên món ăn
y_test_names = le.inverse_transform(y_test_enc)
y_pred_names = le.inverse_transform(y_pred)

# Tính toán các chỉ số
acc = accuracy_score(y_test_names, y_pred_names)
prec = precision_score(y_test_names, y_pred_names, average='weighted')
rec = recall_score(y_test_names, y_pred_names, average='weighted')
f1 = f1_score(y_test_names, y_pred_names, average='weighted')

print("\n--- KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\n--- CHI TIẾT TỪNG LỚP (CLASSIFICATION REPORT) ---")
print(classification_report(y_test_names, y_pred_names))



In [ ]:
# --- 4. VẼ MA TRẬN NHẦM LẪN (CONFUSION MATRIX) DẠNG PHẦN TRĂM ---
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Tính ma trận nhầm lẫn
cm = confusion_matrix(y_test_names, y_pred_names, labels=le.classes_)

# Chuyển đổi sang phần trăm (Normalize theo hàng - True Labels)
# Cộng thêm 1e-9 để tránh lỗi chia cho 0
cm_percent = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(30, 30))
plt.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Greens) # Dùng màu Green cho RF để khác biệt xíu
plt.title("Confusion Matrix (%) - Random Forest")
plt.colorbar()

tick_marks = np.arange(len(le.classes_))
plt.xticks(tick_marks, le.classes_, rotation=90, fontsize=12)
plt.yticks(tick_marks, le.classes_, fontsize=12)

# Hiển thị số phần trăm trong từng ô
thresh = cm_percent.max() / 2
for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        # Chỉ hiển thị nếu giá trị > 0.1% để biểu đồ thoáng hơn
        if cm_percent[i, j] > 0.001:
            plt.text(j, i, f"{cm_percent[i, j]*100:.1f}%",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm_percent[i, j] > thresh else "black",
                     fontsize=10)

plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.tight_layout()
plt.show()